Alissa Rivero — Week 2 Mini-Assignment, Question 2. This is the class Rust notebook (`rust_vs_python_intro.ipynb`) with extra cells at the bottom that experiment with ownership.

# Why is Rust underneath so many fast Python tools?

We just used Polars from Python. One reason it can feel fast is that much of the engine underneath is written in Rust.


Pick the **Rust** kernel (Select Another Kernel, Jupyter Kernel, Rust). If `let` is a `SyntaxError`, you are still on Python. Environment setup is in `SETUP.md`.

A few cells are supposed to fail. Read the one sentence that matters, then keep going. After the takeaway there is one optional extra about memory.

## First, Rust is still just programming

Before the parts that feel foreign, notice how much of this you already know from Python.


### Hello, Rust

A real Rust program lives in a `.rs` file and starts at `fn main`. Open `rust/examples/01_hello.rs` if you want to see that. You do not need to compile it.

This notebook is a little different. The kernel will not call `main` for you, so the last line of the next cell does.


In [2]:
// A `.rs` file starts here. `main` is the function the program runs.
fn main() {
    let engine = "Rust"; // `let` names a value
    println!("Hello from {engine}.");
}

main(); // the notebook does not call `main` for you, so we call it here


Hello from Rust.


In a `.rs` file you would stop at the closing brace. `rustc` calls `main` for you.

The kernel compiled that cell, then ran it. A typo would have stopped it before anything printed.

From here on we skip `fn main` and write the lines directly, the same way you typed Python in part 1.


Your turn. Fill in a name, a movie you like, and how many stars you would give it. The `{name}` pieces insert whatever you put in the quotes.


In [3]:
fn main() {
    let name = "alissa";
    let movie = "barbie";
    let stars = 5;

    println!("{name}'s review");
    println!("  {movie}");
    println!("  {stars} / 5 stars");
}
main();


alissa's review


  barbie


  5 / 5 stars


### Conditions look familiar

Same idea as Python. No colon after the condition, the body goes in `{ }`, and you write `else if` instead of `elif`.

Run this one. We are still talking about a movie rating.


In [4]:
let rating = 4.5;

if rating >= 4.0 {
    println!("{rating} stars: high");
} else if rating >= 3.0 {
    println!("{rating} stars: in the middle");
} else {
    println!("{rating} stars: low");
}


4.5 stars: high


()

Your turn

A movie has `n` ratings. Print `keep` if it has at least 1000, otherwise print `skip`. That is the same cutoff we used in part 1.

Start with `n = 3212` (movie 318). Then try a number under 1000.


In [5]:
let n = 3212;

if n >= 1000 {
    println!("keep")
} else {
    println!("skip")
}

keep


()

### Loops look familiar too

`0..3` is a range: 0, 1, 2. The 3 is not included. You can also walk a list of ratings, same idea as `for rating in ratings:` in Python.


In [6]:
for n in 0..3 { // 0, 1, 2
    println!("pass {n}");
}

let ratings = [4.5, 3.0, 5.0];
for rating in ratings {
    println!("rating = {rating:.1}");
}


pass 0


pass 1


pass 2


rating = 4.5


rating = 3.0


rating = 5.0


()

Your turn. Count how many of these ratings are 4.0 or higher. `high` should print 3. Put an `if` inside the loop.


In [7]:
let ratings = [4.5, 3.0, 5.0, 2.0, 5.0];
let mut high = 0;
for rating in ratings {
    if rating >= 4.0 {
        high += 1}
    // if this rating is high, add 1 to high
}
println!("high ratings: {high}");


high ratings: 3


## Now Rust starts being different

So far Rust mostly looks like Python with braces. This is where the languages start making very different choices.

### Values stay put unless we say otherwise

In Python you can rebind a name whenever you like. `min_ratings = 1000` and later `min_ratings = 2000` is fine.

Rust asks first. A plain `let` holds still. Write `let mut` when the value is supposed to change.

Here the cutoff stays put. The running row count does not.


In [8]:
// No `mut`: this cutoff is not allowed to change.
let min_ratings = 1000;
println!("keep movies with at least {min_ratings} ratings");

// `mut` means this one is allowed to change. Here, a running count.
let mut rows_read = 0;
for _ in 0..3 { // `_` means we do not use the loop counter
    rows_read += 100_000;
    println!("rows_read = {rows_read}");
}


keep movies with at least 1000 ratings


rows_read = 100000


rows_read = 200000


rows_read = 300000


()

Don't run the next cell yet. In Python this would just work: set the cutoff to 1000, then change it to 2000.

What do you think Rust will do?


In [9]:
let min_ratings = 1000;
println!("keep movies with at least {min_ratings} ratings");

min_ratings = 2000;
println!("keep movies with at least {min_ratings} ratings");


Error: cannot assign twice to immutable variable `min_ratings`

The important line in the error is "cannot assign twice to immutable variable". It names both lines and suggests `mut`.

Your turn. Add `mut` on the `let` line, then uncomment the assignment. If you uncomment first, you get the same error again. That is useful.


In [10]:
let mut min_ratings = 1000; // add `mut` on this line
println!("keep movies with at least {min_ratings} ratings");

min_ratings = 2000; // uncomment this after you add `mut`
println!("keep movies with at least {min_ratings} ratings");


keep movies with at least 1000 ratings


keep movies with at least 2000 ratings


Python never asked which values were supposed to hold still. Nothing in those four lines was "wrong", and that is the problem.

Rust now knows the cutoff was meant to stay put, unless we say otherwise. Next question: who owns a list of ratings?


### Who owns this data?

In Python, `b = a` is a second name for the same list. Both names share it.

In Rust, a list on the heap has one owner. `let moved = ratings` can hand the list over. After that, `ratings` is empty-handed.

You either move it, or you `.clone()` it and pay for a second list.


In [11]:
// `vec!` is a list. These are three movie ratings from part 1.
let ratings = vec![4.5, 3.0, 5.0];
println!("ratings = {ratings:?}"); // `:?` prints the list in a readable way


ratings = [4.5, 3.0, 5.0]


Don't run this yet. After `let moved = ratings`, can you still print both names?


In [12]:
let ratings = vec![4.5, 3.0, 5.0];
let moved = ratings;
println!("{ratings:?} {moved:?}");


Error: borrow of moved value: `ratings`

The error is "borrow of moved value: ratings". The list went to `moved`. Using `ratings` after that is the problem.

The compiler also offers `.clone()`. That is the expensive option: a real second list. The next cell does that so both names can live.


In [13]:
let ratings = vec![4.5, 3.0, 5.0];
let copy = ratings.clone(); // a second list, so both names can live
let moved = ratings; // this move is fine: we still have `copy`
println!("moved = {moved:?}");
println!("copy  = {copy:?}");


moved = [4.5, 3.0, 5.0]


copy  = [4.5, 3.0, 5.0]


Your turn. Delete `.clone()` and run. You should see the same kind of error. Put `.clone()` back so both lines print.

Moving every time would get old. What if we just want to look at the ratings without taking them?


In [14]:
let ratings = vec![4.5, 3.0, 5.0];
let copy = ratings.clone(); // delete `.clone()`, run, then put it back
println!("ratings = {ratings:?}");
println!("copy    = {copy:?}");


ratings = [4.5, 3.0, 5.0]


copy    = [4.5, 3.0, 5.0]


### Borrowing instead of copying

Sometimes we do not want to hand the data over. We just want to let another piece of code use it for a moment.

`&ratings` is a loan. Many readers at once is fine. If somebody is going to write, they need the list to themselves.

Many readers, or one writer. Not both.


In [15]:
let mut ratings = vec![5, 3, 4, 4];
{
    // `&` lends the list for reading. Two readers at once is allowed.
    let first = &ratings;
    let second = &ratings;
    println!("two readers: {first:?} and {second:?}");
} // the loans end here, so a write is allowed again

ratings.push(2); // add a 2-star rating
println!("after a write: {ratings:?}");


two readers: [5, 3, 4, 4] and [5, 3, 4, 4]


after a write: [5, 3, 4, 4, 2]


These rules can feel annoying in toy examples. Now let's see the kind of bug they are trying to prevent.

## Why all these rules?

### A bug Python will happily run

Don't run this yet. In Python, removing items from a list while you loop over it often "works". It can also silently skip an item.

Start with `[1, 2, 2, 3, 4, 5]`, drop every 2 while looping, and one 2 survives. No error.

The next cell is the same idea. The `&ratings` is the loan from a minute ago: we are reading the list. Then we try to change it in the same loop.

What do you think Rust will do?


In [16]:
// This cell is supposed to fail. Same idea as Python's list.remove while looping.
let mut ratings = vec![1, 2, 2, 3, 4, 5];

for rating in ratings {
    if rating == 2 {
        ratings.remove(1);
    }
}

println!("{ratings:?}");


Error: borrow of moved value: `ratings`

The important sentence is "cannot borrow `ratings` as mutable because it is also borrowed as immutable". We are reading the list and trying to change it at the same time.

Python runs the same idea and can give the wrong answer, with no error. `ratings.remove(2)` on `[1, 2, 2, 3, 4, 5]` leaves `[1, 2, 3, 4, 5]`. The second 2 was skipped because the list shifted under the loop.

That is what the earlier rules were for. Rust made us state who was reading and who was writing, then refused the program that mixed them.


## So what should you remember?

Basic Rust is recognizable if you know Python. `fn main`, `if`, and `for` are the same ideas with different spelling.

Then Rust asks us to be explicit. A plain `let` does not change. A list has one owner. A loan is either shared-and-read or exclusive-and-write.

Python gives more freedom. Some mistakes then show up only when the line runs, or they change the answer and never raise.

Rust rejects some of those programs before they run. That is not a reason to drop Python. Tools like Polars can keep the Python interface and still use this kind of compiled Rust underneath.


### Practical note

You do not need to write Rust to get this. Polars is a Python package. You `import polars`, write Python, and a compiled Rust engine does the heavy work.

As a data scientist you will live in Python most of the time. The useful move is to notice when a library is a thin wrapper around Rust (or C, or C++) and let that engine do the scan, the join, the group-by.

`pip install polars` is that wrapper. The rules we just saw are why the engine underneath can be strict and fast, while the notebook you type in stays Python.


## Optional: where the memory goes

You can stop here. This extra is only if you want to see a value get freed.

Rust has no garbage collector. The compiler inserts the free at compile time. `Drop` prints at the moment a value is released, so you can watch it.

Look for `FREE 64 MB` before the line "back in the outer block". After it runs, change `64` to `16` and run again. The `FREE` line should match.


In [17]:
// A Buffer is a block of bytes we can watch being freed.
struct Buffer {
    data: Vec<u8>, // the actual bytes
}

fn make_buffer(megabytes: usize) -> Buffer {
    println!("  allocate {megabytes} MB");
    Buffer {
        data: vec![0; megabytes * 1024 * 1024],
    }
}

// `Drop` runs automatically when a Buffer goes out of scope. No `free()` call.
impl Drop for Buffer {
    fn drop(&mut self) {
        let megabytes = self.data.len() / 1024 / 1024;
        println!("  FREE {megabytes} MB");
    }
}

println!("enter outer block");
{
    println!("  enter inner scope");
    let _scratch = make_buffer(64); // create it only to watch it die
    println!("  inner scope is about to end");
} // scratch is freed on this brace
println!("back in the outer block: that memory is already gone");


enter outer block


  enter inner scope


  allocate 64 MB


  inner scope is about to end


  FREE 64 MB


back in the outer block: that memory is already gone


CPython frees when the last name is gone, by counting references at runtime. Two objects that point at each other need a second collector.

In Rust you do not stumble into a cycle. Shared ownership is a type you ask for by name (`Rc`). The compiler already knew when that 64 MB block could go away.


## Extra: my ownership experiments

The cells above are the class notebook. These extra cells use a tiny list of gut taxa to try the same three ideas again: **move**, **borrow**, and **one writer**.

I changed the examples, caused a couple of compiler errors on purpose, then wrote a version that compiles.


### Move: one owner for a list of taxa

`let other = taxa` hands the `Vec` to `other`. After that, `taxa` cannot be used. The next cell is supposed to fail.


In [18]:
let taxa = vec![
    "Bifidobacterium",
    "Faecalibacterium",
    "Escherichia coli",
];
let other = taxa; // move: taxa no longer owns the list
println!("still mine? {taxa:?}"); // should fail: borrow of moved value


Error: borrow of moved value: `taxa`

The error is **borrow of moved value: `taxa`**. The list now belongs to `other`.

Two ways to keep using the names: **clone** (a second list) or **borrow** (`&taxa`, a loan).


In [19]:
let taxa = vec![
    String::from("Bifidobacterium"),
    String::from("Faecalibacterium"),
    String::from("Escherichia coli"),
];

// clone: a real second list, so both names can live
let backup = taxa.clone();
let moved = taxa;
println!("moved  = {moved:?}");
println!("backup = {backup:?}");


moved  = ["Bifidobacterium", "Faecalibacterium", "Escherichia coli"]


backup = ["Bifidobacterium", "Faecalibacterium", "Escherichia coli"]


### Borrow: look without taking

`count_owned` takes the list. After the call, the caller has nothing. `count_borrowed` only asks for `&Vec<String>`, so the caller still owns the taxa.


In [20]:
fn count_owned(taxa: Vec<String>) -> usize {
    taxa.len()
}

fn count_borrowed(taxa: &Vec<String>) -> usize {
    taxa.len()
}

let taxa = vec![
    String::from("Roseburia"),
    String::from("Ruminococcus"),
];

let n_borrowed = count_borrowed(&taxa);
println!("borrowed count = {n_borrowed}; still own taxa = {taxa:?}");

let n_owned = count_owned(taxa); // this call takes the list
println!("owned count = {n_owned}");
// println!("{taxa:?}"); // uncomment to see: taxa was moved into count_owned


borrowed count = 2; still own taxa = ["Roseburia", "Ruminococcus"]


owned count = 2


### Many readers, or one writer

Two `&` loans at once are fine. A `&mut` loan has to be alone. The next cell is supposed to fail: a reader and a writer at the same time.


In [21]:
let mut taxa = vec![String::from("Prevotella"), String::from("Bacteroides")];
let reader = &taxa;
taxa.push(String::from("Desulfovibrio")); // write while a reader is still out
println!("reader still sees {reader:?}");


Error: The variable `reader` contains a reference with a non-static lifetime so
can't be persisted. You can prevent this error by making sure that the
variable goes out of scope - i.e. wrapping the code in {}.

Error: cannot borrow `taxa` as mutable because it is also borrowed as immutable

The important line is **cannot borrow `taxa` as mutable because it is also borrowed as immutable**.

Drop the reader first, then write:


In [22]:
let mut taxa = vec![String::from("Prevotella"), String::from("Bacteroides")];
{
    let reader = &taxa;
    println!("readers only: {reader:?}");
} // reader loan ends here

taxa.push(String::from("Desulfovibrio"));
println!("after exclusive write: {taxa:?}");


readers only: ["Prevotella", "Bacteroides"]


after exclusive write: ["Prevotella", "Bacteroides", "Desulfovibrio"]


### What I changed and what I learned

- **Your-turn cells above:** filled in my name, a movie, the 1000-rating cutoff, a loop count, `mut` on the cutoff, and `.clone()` on a ratings list.
- **These extra cells:** same rules, but with taxon names from the ASD analysis so the moves and loans are about data I already used in Python.
- **Move vs borrow:** `count_owned(taxa)` empties the caller; `count_borrowed(&taxa)` does not. That is the difference Polars can rely on in Rust: the engine knows when a buffer is unique and when it is only being read.
- **The failed cells are the point.** Python would let me keep both names after `b = a`, or edit a list while looping. Rust stops those before they run.
